# TerraWatch — Damage Assessment Pipeline
## ESRGAN Super-Resolution + YOLOv8n Fine-tuning on xBD Dataset

**Run on Kaggle with GPU T4 x2 accelerator**

Pipeline:
1. Download & preprocess xBD building damage dataset
2. Establish YOLOv8n baseline (raw low-res images)
3. Apply Real-ESRGAN 4x upscaling
4. Fine-tune YOLOv8n on enhanced images
5. Compare accuracy — this is our headline result
6. Export best model for FastAPI serving

**Our claim**: ESRGAN enhancement before YOLOv8n improves damage detection mAP by ~12-18% on low-resolution satellite imagery

In [ ]:
# ── ENVIRONMENT SETUP (RUN FIRST EVERY SESSION) ──────────────────────────────

print("Installing required packages...")

%pip -q install ultralytics
%pip -q install realesrgan basicsr

print("✅ Dependencies installed")

In [ ]:
# ── PREFLIGHT CHECK (run this at the very start) ─────────────────────────────

import sys, types, time
import numpy as np
import cv2
import torch
import torchvision
import torchvision.transforms.functional as F
from pathlib import Path

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

# --- Kaggle-safe shim for basicsr/realesrgan ---
if "torchvision.transforms.functional_tensor" not in sys.modules:
    functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
    functional_tensor.rgb_to_grayscale = F.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

# YOLO import test
try:
    from ultralytics import YOLO
    _ = YOLO("yolov8n.pt")
    print("✅ Ultralytics/YOLO import OK")
except Exception as e:
    raise RuntimeError("❌ YOLO import failed") from e

# ESRGAN import test
try:
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    print("✅ RealESRGAN imports OK")
except Exception as e:
    raise RuntimeError("❌ RealESRGAN import failed") from e

# weights download test (fast)
weights_path = Path("/kaggle/working/RealESRGAN_x4plus.pth")
if not weights_path.exists():
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O /kaggle/working/RealESRGAN_x4plus.pth
print("✅ Weights file exists:", weights_path.exists(), "| size:", weights_path.stat().st_size)

# minimal ESRGAN inference test on a dummy image (fast, no dataset needed)
rrdb_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path=str(weights_path),
    model=rrdb_model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),
    gpu_id=0 if torch.cuda.is_available() else None
)

dummy = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
t0 = time.time()
out, _ = upsampler.enhance(dummy, outscale=4)
print(f"✅ ESRGAN enhance OK: {dummy.shape} -> {out.shape} in {time.time()-t0:.2f}s")

print("✅ PREFLIGHT PASSED — safe to run the notebook")

In [ ]:
# ── INSTALL DEPENDENCIES ──────────────────────────────────────────────────────
!pip install -q ultralytics basicsr facexlib gfpgan
!pip install -q git+https://github.com/xinntao/Real-ESRGAN.git
!pip install -q supervision roboflow

import os, json, shutil, random, time
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import supervision as sv

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── DATASET SETUP ─────────────────────────────────────────────────────────────
# Option A: xBD dataset via Kaggle (add dataset: zabirauf/xview2-xbd-dataset)
# Option B: Use this curated subset we prepared from xBD

# xBD damage classes:
# 0: no-damage      (intact building)
# 1: minor-damage   (cracked walls, broken windows)
# 2: major-damage   (partial collapse, heavy debris)
# 3: destroyed      (complete collapse)

DAMAGE_CLASSES = {
    0: 'no-damage',
    1: 'minor-damage', 
    2: 'major-damage',
    3: 'destroyed'
}

BASE_DIR = Path('/kaggle/working/terrawatch')
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
BASE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# Check if xBD dataset is attached
XBD_PATH = Path('/kaggle/input/datasets/tunguz/xview2-challenge-dataset-train-and-test')
if XBD_PATH.exists():
    print(f'xBD dataset found: {XBD_PATH}')
    USE_REAL_DATA = True
else:
    print('xBD not found — generating synthetic training data')
    print('Attach dataset: kaggle datasets download -d zabirauf/xview2-xbd-dataset')
    USE_REAL_DATA = False

In [ ]:
# ── XBD / xView2 DATA PREPROCESSING ───────────────────────────────────────────
# Reads labeled TRAIN data (images + JSON polygons) from Kaggle INPUT,
# converts polygons -> YOLO bboxes, writes YOLO dataset into Kaggle WORKING.

import json
import random
import shutil
from pathlib import Path

import cv2
from shapely.geometry import shape

from shapely.geometry import shape
from shapely import wkt as shapely_wkt

def xbd_json_to_yolo(json_path: Path, img_w: int, img_h: int, debug: bool = False):
    """
    Convert xView2/xBD JSON annotation to YOLO bbox format:
    each line: "cls cx cy w h" normalized to 0..1

    Supports:
      - data["features"]["xy"] as list of features
      - feature geometry stored as GeoJSON dict in feature["geometry"]
      - OR WKT string in feature["wkt"]
    """
    SUBTYPE_MAP = {
        "no-damage": 0,
        "minor-damage": 1,
        "major-damage": 2,
        "destroyed": 3,
        "un-classified": 0,
        None: 0,
    }

    try:
        with open(json_path, "r") as f:
            data = json.load(f)
    except Exception:
        return []

    # --- Robustly extract features list ---
    feats = []
    fobj = data.get("features", None)
    if isinstance(fobj, dict) and isinstance(fobj.get("xy", None), list):
        feats = fobj["xy"]
    elif isinstance(fobj, list):
        feats = fobj

    if not feats:
        if debug:
            print("DEBUG: no features found")
        return []

    # Debug counters
    c_total = 0
    c_no_geom = 0
    c_parse_fail = 0
    c_small = 0
    c_ok = 0

    yolo_lines = []

    for feat in feats:
        if not isinstance(feat, dict):
            continue
        c_total += 1

        props = feat.get("properties", {}) or {}
        subtype = props.get("subtype", "no-damage")
        cls = SUBTYPE_MAP.get(str(subtype).lower() if subtype is not None else None, 0)

        # Parse geometry: prefer GeoJSON; fallback to WKT
        geom_obj = None
        try:
            if feat.get("geometry") is not None:
                geom_obj = shape(feat["geometry"])  # GeoJSON dict
            elif feat.get("wkt") is not None:
                geom_obj = shapely_wkt.loads(feat["wkt"])  # WKT string
            else:
                c_no_geom += 1
                continue
        except Exception:
            c_parse_fail += 1
            continue

        # Convert bounds to YOLO
        try:
            x1, y1, x2, y2 = geom_obj.bounds
            cx = ((x1 + x2) / 2.0) / img_w
            cy = ((y1 + y2) / 2.0) / img_h
            w  = (x2 - x1) / img_w
            h  = (y2 - y1) / img_h

            # Basic sanity filters
            if w <= 0.005 or h <= 0.005:
                c_small += 1
                continue
            if not (0 <= cx <= 1 and 0 <= cy <= 1):
                c_parse_fail += 1
                continue

            yolo_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            c_ok += 1
        except Exception:
            c_parse_fail += 1
            continue

    if debug:
        print(f"DEBUG feats={len(feats)} total={c_total} ok={c_ok} no_geom={c_no_geom} parse_fail={c_parse_fail} small={c_small}")

    return yolo_lines

def prepare_xbd_dataset(xbd_root, output_dir, max_images=2000, seed=42):
    """
    Prepare xView2 challenge TRAIN data in YOLO format.
    IMPORTANT: Uses /train/train/{images,labels}. Test has no labels.
    """
    xbd_root = Path(xbd_root)
    output_dir = Path(output_dir)

    train_images_dir = xbd_root / "train" / "train" / "images"
    train_labels_dir = xbd_root / "train" / "train" / "labels"

    if not train_images_dir.exists() or not train_labels_dir.exists():
        raise FileNotFoundError(
            f"Expected train dirs not found.\nimages: {train_images_dir}\nlabels: {train_labels_dir}"
        )

    # Start fresh each time to avoid stale empty folders
    if output_dir.exists():
        shutil.rmtree(output_dir)

    for split in ["train", "val", "test"]:
        (output_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (output_dir / split / "labels").mkdir(parents=True, exist_ok=True)

    # Train images are typically "*_post_disaster.*"
    exts = {".png", ".jpg", ".jpeg"}
    all_post_images = sorted([
        p for p in train_images_dir.rglob("*_post_disaster.*")
        if p.suffix.lower() in exts
    ])[:max_images]

    random.seed(seed)
    random.shuffle(all_post_images)

    n = len(all_post_images)
    print("Found TRAIN post_disaster images:", n)

    if n == 0:
        print("No TRAIN post_disaster images found under:", train_images_dir)
        return output_dir

    # Quick one-file test so you can immediately see whether boxes are being produced
    sample_img = all_post_images[0]
    sample_json = train_labels_dir / (sample_img.stem + ".json")
    img0 = cv2.imread(str(sample_img))
    h0, w0 = img0.shape[:2]
    lines0 = xbd_json_to_yolo(sample_json, w0, h0)

    print("Sample image:", sample_img.name)
    print("Expected JSON exists?:", sample_json.exists(), "|", sample_json.name)
    print("Sample boxes produced:", len(lines0))
    if len(lines0) > 0:
        print("First 3 YOLO lines:", lines0[:3])

    splits = {
        "train": all_post_images[: int(n * 0.7)],
        "val":   all_post_images[int(n * 0.7): int(n * 0.85)],
        "test":  all_post_images[int(n * 0.85):],
    }

    stats = {"total": 0, "missing_json": 0, "empty_yolo": 0, "bad_img": 0}

    for split, images in splits.items():
        for img_path in images:
            json_path = train_labels_dir / (img_path.stem + ".json")
            if not json_path.exists():
                stats["missing_json"] += 1
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                stats["bad_img"] += 1
                continue

            h, w = img.shape[:2]
            yolo_lines = xbd_json_to_yolo(json_path, w, h)
            if not yolo_lines:
                stats["empty_yolo"] += 1
                continue

            shutil.copy2(img_path, output_dir / split / "images" / img_path.name)
            (output_dir / split / "labels" / (img_path.stem + ".txt")).write_text("\n".join(yolo_lines))
            stats["total"] += 1

    yaml_content = f"""path: {output_dir}
train: train/images
val: val/images
test: test/images

nc: 4
names: ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
"""
    (output_dir / "damage.yaml").write_text(yaml_content)

    print(f"Dataset prepared: {stats['total']} images")
    print(f"Skipped: missing_json={stats['missing_json']}, empty_yolo={stats['empty_yolo']}, bad_img={stats['bad_img']}")
    print(f"Splits planned: train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")

    return output_dir


# Run conversion if real data is enabled
if USE_REAL_DATA:
    dataset_dir = prepare_xbd_dataset(XBD_PATH, DATA_DIR / "xbd_yolo", max_images=2000)
else:
    print("Using synthetic data — see next cell")

# Verify output exists
print("\n=== YOLO output check ===")
!find {dataset_dir}/train/images -type f | head
!find {dataset_dir}/train/images -type f | wc -l
!find {dataset_dir}/train/labels -type f | head
!find {dataset_dir}/train/labels -type f | wc -l
!ls -lah {dataset_dir} | head

In [ ]:
# ── SYNTHETIC DATA FALLBACK ───────────────────────────────────────────────────
# If real data isn't available, generate synthetic satellite-like images
# with building footprints and damage labels for pipeline testing.

import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def generate_synthetic_dataset(output_dir, n_train=800, n_val=200, n_test=100):
    """Generate synthetic satellite imagery for pipeline testing."""
    output_dir = Path(output_dir)
    for split in ["train", "val", "test"]:
        (output_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (output_dir / split / "labels").mkdir(parents=True, exist_ok=True)

    COLORS = {
        0: (180, 180, 180),  # no-damage
        1: (140, 120, 100),  # minor-damage
        2: (100, 80,  60),   # major-damage
        3: (60,  40,  30),   # destroyed
    }

    def make_image(path_img, path_lbl, n_buildings=15, quality=100):
        W, H = 640, 640
        img = np.random.randint(45, 75, (H, W, 3), dtype=np.uint8)

        labels = []
        for _ in range(n_buildings):
            bw = random.randint(30, 90)
            bh = random.randint(30, 90)
            bx = random.randint(0, W - bw)
            by = random.randint(0, H - bh)

            cls = random.choices([0, 1, 2, 3], weights=[0.4, 0.25, 0.2, 0.15])[0]
            color = COLORS[cls]
            noise = np.random.randint(-20, 20, (bh, bw, 3))
            patch = np.clip(np.array(color, dtype=np.int32) + noise, 0, 255).astype(np.uint8)
            img[by:by+bh, bx:bx+bw] = patch

            if cls >= 2:
                for _ in range(cls * 3):
                    rx = random.randint(bx, bx + bw - 5)
                    ry = random.randint(by, by + bh - 5)
                    rw = random.randint(3, 15)
                    rh = random.randint(3, 15)
                    img[ry:ry+rh, rx:rx+rw] = np.random.randint(20, 80, (rh, rw, 3), dtype=np.uint8)

            cx = (bx + bw / 2) / W
            cy = (by + bh / 2) / H
            labels.append(f"{cls} {cx:.6f} {cy:.6f} {bw/W:.6f} {bh/H:.6f}")

        if quality < 100:
            scale = quality / 100
            small = cv2.resize(img, (int(W*scale), int(H*scale)))
            img = cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR)

        cv2.imwrite(str(path_img), img, [cv2.IMWRITE_JPEG_QUALITY, quality])
        Path(path_lbl).write_text("\n".join(labels))

    for split, n in [("train", n_train), ("val", n_val), ("test", n_test)]:
        for i in range(n):
            quality = random.choice([100, 100, 60, 40]) if split == "train" else 40
            make_image(
                output_dir / split / "images" / f"img_{i:04d}.jpg",
                output_dir / split / "labels" / f"img_{i:04d}.txt",
                quality=quality,
            )
        print(f"{split}: {n} images generated")

    yaml_content = f"""path: {output_dir}
train: train/images
val: val/images
test: test/images
nc: 4
names: ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
"""
    (output_dir / "damage.yaml").write_text(yaml_content)
    print(f"Synthetic dataset ready at {output_dir}")
    return output_dir


# If real dataset prep failed or real data disabled, generate synthetic
if (not USE_REAL_DATA) or (dataset_dir is None) or (not (Path(dataset_dir) / "train" / "images").exists()):
    dataset_dir = generate_synthetic_dataset(DATA_DIR / "xbd_yolo")

# Quick visualization (safe even if empty)
train_img_dir = Path(dataset_dir) / "train" / "images"
sample_imgs = list(train_img_dir.glob("*.jpg"))[:4] + list(train_img_dir.glob("*.png"))[:4]

if len(sample_imgs) == 0:
    print(f"No sample images found in {train_img_dir} (dataset may be empty).")
else:
    ncols = min(4, len(sample_imgs))
    fig, axes = plt.subplots(1, ncols, figsize=(16, 4))
    if ncols == 1:
        axes = [axes]
    for ax, p in zip(axes, sample_imgs[:ncols]):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.stem[:20], fontsize=9)
        ax.axis("off")
    plt.suptitle("Sample training images", fontweight="bold")
    plt.tight_layout()
    plt.savefig(BASE_DIR / "sample_images.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── BASELINE: YOLOV8n WITHOUT ESRGAN ─────────────────────────────────────────
# Train on raw images first — this is our baseline to beat

print('='*50)
print('STEP 1: Baseline YOLOv8n (no super-resolution)')
print('='*50)

baseline_model = YOLO('yolov8n.pt')

baseline_results = baseline_model.train(
    data=str(dataset_dir / 'damage.yaml'),

    epochs=150,        # ← increase
    patience=50,       # ← add (early stop if no improvement)

    imgsz=640,
    batch=16,

    lr0=1e-3,
    lrf=0.01,
    momentum=0.937,
    weight_decay=5e-4,
    warmup_epochs=3,

    # Augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    flipud=0.3, fliplr=0.5,
    mosaic=1.0,

    project=str(MODELS_DIR),
    name='baseline_yolov8n',
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    verbose=False,
)

# Get baseline metrics
baseline_metrics = baseline_model.val(
    data=str(dataset_dir / 'damage.yaml'),
    split='test'
)
baseline_map50   = baseline_metrics.box.map50
baseline_map5095 = baseline_metrics.box.map
print(f'\nBaseline mAP@50:    {baseline_map50:.4f}')
print(f'Baseline mAP@50-95: {baseline_map5095:.4f}')

# ── F1 SCORE (Baseline) ─────────────────────────────────────

precision = baseline_metrics.box.mp
recall    = baseline_metrics.box.mr

baseline_f1 = 2 * precision * recall / (precision + recall + 1e-9)

print(f'Baseline Precision: {precision:.4f}')
print(f'Baseline Recall:    {recall:.4f}')
print(f'Baseline F1 Score:  {baseline_f1:.4f}')

In [ ]:
# ── REAL-ESRGAN ENHANCEMENT ───────────────────────────────────────────────────
# Upscale ALL training + test images 4x with RealESRGAN
# This simulates enhancing low-resolution satellite imagery

import sys
import types
import time
from pathlib import Path

import cv2
import torch
import torchvision.transforms.functional as F

# ---- Kaggle-safe compatibility shim ----
# basicsr (pulled by realesrgan) may import:
#   from torchvision.transforms.functional_tensor import rgb_to_grayscale
# which doesn't exist in some torchvision versions.
if "torchvision.transforms.functional_tensor" not in sys.modules:
    functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
    functional_tensor.rgb_to_grayscale = F.rgb_to_grayscale
    sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

# Now these imports should work reliably
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

print('='*50)
print('STEP 2: RealESRGAN 4x Enhancement')
print('='*50)

# ---------- QUICK SMOKE TEST (fast) ----------
# This prevents you from wasting time enhancing thousands of images if the stack is broken.
print("torch:", torch.__version__)
import torchvision
print("torchvision:", torchvision.__version__)

# Download pretrained RealESRGAN weights (only if missing)
weights_path = Path("/kaggle/working/RealESRGAN_x4plus.pth")
if not weights_path.exists():
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O /kaggle/working/RealESRGAN_x4plus.pth
print("Weights:", weights_path, "| exists?", weights_path.exists(), "| size:", weights_path.stat().st_size)

# Initialize ESRGAN
rrdb_model = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=4
)

# NOTE: RealESRGANer uses gpu_id in many versions; some accept device.
# We'll pass gpu_id for compatibility.
upsampler = RealESRGANer(
    scale=4,
    model_path=str(weights_path),
    model=rrdb_model,
    tile=400,       # tile processing — memory efficient for large images
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),  # FP16 on GPU
    gpu_id=0 if torch.cuda.is_available() else None,
)

# Smoke test on ONE image (fast, proves enhance() works)
test_imgs = list((dataset_dir / "train" / "images").glob("*.png"))[:1] + list((dataset_dir / "train" / "images").glob("*.jpg"))[:1]
if len(test_imgs) == 0:
    raise RuntimeError("No training images found to smoke-test ESRGAN. Check dataset_dir/train/images.")
test_img_path = test_imgs[0]
test_img = cv2.imread(str(test_img_path), cv2.IMREAD_UNCHANGED)
if test_img is None:
    raise RuntimeError(f"Failed to read smoke-test image: {test_img_path}")

t0 = time.time()
out, _ = upsampler.enhance(test_img, outscale=4)
print(f"✅ ESRGAN smoke test OK: {test_img_path.name} -> {out.shape} in {time.time()-t0:.2f}s")
print('ESRGAN model loaded')


def enhance_directory(src_dir, dst_dir, upsampler, target_size=640):
    """
    Enhance all images in src_dir with ESRGAN, resize to target_size.
    We resize BACK to 640 after 4x upscale so YOLO input size is consistent.
    """
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    images = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.png'))
    print(f'Enhancing {len(images)} images from {src_dir}...')

    t0 = time.time()
    for i, img_path in enumerate(images):
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
        if img is None:
            continue

        try:
            enhanced, _ = upsampler.enhance(img, outscale=4)
            # Resize back to 640 (YOLO standard input)
            enhanced = cv2.resize(enhanced, (target_size, target_size),
                                  interpolation=cv2.INTER_LANCZOS4)
        except RuntimeError as e:
            print(f'  ESRGAN failed on {img_path.name}: {e} — using bicubic')
            enhanced = cv2.resize(img, (target_size, target_size),
                                  interpolation=cv2.INTER_CUBIC)

        out_path = dst_dir / img_path.name
        cv2.imwrite(str(out_path), enhanced)

        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 1e-6)
            eta = (len(images) - i - 1) / max(rate, 1e-6)
            print(f'  {i+1}/{len(images)} | {rate:.1f} img/s | ETA {eta:.0f}s')

    print(f'  Done in {time.time()-t0:.1f}s')


# Create enhanced dataset directory (labels are same, only images change)
enhanced_dir = DATA_DIR / 'xbd_enhanced'
for split in ['train', 'val', 'test']:
    # Enhance images
    enhance_directory(
        dataset_dir / split / 'images',
        enhanced_dir / split / 'images',
        upsampler
    )

    # Copy labels (safer than symlink in Kaggle)
    src_lbl = dataset_dir / split / 'labels'
    dst_lbl = enhanced_dir / split / 'labels'
    dst_lbl.mkdir(parents=True, exist_ok=True)
    for lbl in src_lbl.glob("*.txt"):
        shutil.copy2(lbl, dst_lbl / lbl.name)

# Write enhanced dataset YAML
(enhanced_dir / 'damage.yaml').write_text(f"""path: {enhanced_dir}
train: train/images
val: val/images
test: test/images
nc: 4
names: ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
""")
print('Enhanced dataset ready:', enhanced_dir)

In [ ]:
# ── ENHANCED: YOLOV8n WITH ESRGAN ────────────────────────────────────────────
print('='*50)
print('STEP 3: YOLOv8n trained on ESRGAN-enhanced images')
print('='*50)

enhanced_model = YOLO('yolov8n.pt')

enhanced_results = enhanced_model.train(
    data=str(enhanced_dir / 'damage.yaml'),

    epochs=150,        # ← increase
    patience=50,       # ← add

    imgsz=640,
    batch=16,

    lr0=1e-3,
    lrf=0.01,
    momentum=0.937,
    weight_decay=5e-4,
    warmup_epochs=3,

    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    flipud=0.3, fliplr=0.5,
    mosaic=1.0,

    project=str(MODELS_DIR),
    name='enhanced_yolov8n',
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    verbose=False,
)

enhanced_metrics = enhanced_model.val(
    data=str(enhanced_dir / 'damage.yaml'),
    split='test'
)
enhanced_map50   = enhanced_metrics.box.map50
enhanced_map5095 = enhanced_metrics.box.map
print(f'\nEnhanced mAP@50:    {enhanced_map50:.4f}')
print(f'Enhanced mAP@50-95: {enhanced_map5095:.4f}')

# ── F1 SCORE (Enhanced) ─────────────────────────────────────

precision_e = enhanced_metrics.box.mp
recall_e    = enhanced_metrics.box.mr

enhanced_f1 = 2 * precision_e * recall_e / (precision_e + recall_e + 1e-9)

print(f'Enhanced Precision: {precision_e:.4f}')
print(f'Enhanced Recall:    {recall_e:.4f}')
print(f'Enhanced F1 Score:  {enhanced_f1:.4f}')

In [ ]:
# ── RESULTS COMPARISON ───────────────────────────────────────────────────────
# This is the headline result for the hackathon presentation

# ---- F1 (derived from YOLO precision/recall) ----
baseline_precision = baseline_metrics.box.mp
baseline_recall    = baseline_metrics.box.mr
baseline_f1 = 2 * baseline_precision * baseline_recall / (baseline_precision + baseline_recall + 1e-9)

enhanced_precision = enhanced_metrics.box.mp
enhanced_recall    = enhanced_metrics.box.mr
enhanced_f1 = 2 * enhanced_precision * enhanced_recall / (enhanced_precision + enhanced_recall + 1e-9)

improvement_f1 = (enhanced_f1 - baseline_f1) / max(baseline_f1, 1e-9) * 100

improvement_map50   = (enhanced_map50 - baseline_map50) / baseline_map50 * 100
improvement_map5095 = (enhanced_map5095 - baseline_map5095) / baseline_map5095 * 100

print('\n' + '='*50)
print('HEADLINE RESULT')
print('='*50)
print(f'Baseline  mAP@50:    {baseline_map50:.4f}')
print(f'ESRGAN    mAP@50:    {enhanced_map50:.4f}  (+{improvement_map50:.1f}%)')
print(f'Baseline  mAP@50-95: {baseline_map5095:.4f}')
print(f'ESRGAN    mAP@50-95: {enhanced_map5095:.4f}  (+{improvement_map5095:.1f}%)')
print(f'Baseline  Precision: {baseline_precision:.4f}')
print(f'Baseline  Recall:    {baseline_recall:.4f}')
print(f'Baseline  F1 Score:  {baseline_f1:.4f}')
print(f'ESRGAN    Precision: {enhanced_precision:.4f}')
print(f'ESRGAN    Recall:    {enhanced_recall:.4f}')
print(f'ESRGAN    F1 Score:  {enhanced_f1:.4f}  (+{improvement_f1:.1f}%)')

# Per-class breakdown
print('\nPer-class AP@50 (Enhanced vs Baseline):')
class_names = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
for i, cls in enumerate(class_names):
    b = baseline_metrics.box.ap50[i] if i < len(baseline_metrics.box.ap50) else 0
    e = enhanced_metrics.box.ap50[i] if i < len(enhanced_metrics.box.ap50) else 0
    print(f'  {cls:16s}: {b:.3f} → {e:.3f}  ({(e-b)/max(b,1e-6)*100:+.1f}%)')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Bar chart: mAP comparison
ax = axes[0]
bars = ax.bar(['Baseline\n(no SR)', 'ESRGAN\nEnhanced'],
               [baseline_map50, enhanced_map50],
               color=['#6b7280', '#dc2626'], width=0.5, edgecolor='white', linewidth=2)
ax.set_ylim(0, min(1.0, max(enhanced_map50, baseline_map50) * 1.3))
ax.set_ylabel('mAP@50', fontsize=12)
ax.set_title('Damage Detection Accuracy\nESRGAN vs Baseline', fontweight='bold')
for bar, val in zip(bars, [baseline_map50, enhanced_map50]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=12)
ax.annotate(f'+{improvement_map50:.1f}%', xy=(1, enhanced_map50),
            xytext=(1.2, enhanced_map50 * 0.95),
            color='#dc2626', fontweight='bold', fontsize=14)
ax.spines[['top','right']].set_visible(False)

# Per-class comparison
ax = axes[1]
x = np.arange(4)
baseline_aps = [baseline_metrics.box.ap50[i] if i < len(baseline_metrics.box.ap50) else 0 for i in range(4)]
enhanced_aps = [enhanced_metrics.box.ap50[i] if i < len(enhanced_metrics.box.ap50) else 0 for i in range(4)]
ax.bar(x - 0.2, baseline_aps, 0.35, label='Baseline', color='#6b7280')
ax.bar(x + 0.2, enhanced_aps, 0.35, label='ESRGAN', color='#dc2626')
ax.set_xticks(x)
ax.set_xticklabels(['No\nDamage', 'Minor', 'Major', 'Destroyed'], fontsize=9)
ax.set_ylabel('AP@50')
ax.set_title('Per-Class Detection\nAccuracy', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# Sample visual comparison
ax = axes[2]
test_imgs = list((dataset_dir / 'test' / 'images').glob('*.jpg'))[:1] + \
            list((dataset_dir / 'test' / 'images').glob('*.png'))[:1]
if test_imgs:
    raw_img = cv2.cvtColor(cv2.imread(str(test_imgs[0])), cv2.COLOR_BGR2RGB)
    enh_path = enhanced_dir / 'test' / 'images' / test_imgs[0].name
    enh_img = cv2.cvtColor(cv2.imread(str(enh_path)), cv2.COLOR_BGR2RGB) if enh_path.exists() else raw_img
    
    combined = np.hstack([
        cv2.resize(raw_img, (300, 300)),
        np.ones((300, 10, 3), dtype=np.uint8) * 255,
        cv2.resize(enh_img, (300, 300))
    ])
    ax.imshow(combined)
    ax.set_title('Raw (left) vs ESRGAN (right)', fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(150, 315, 'Low-res input', ha='center', fontsize=8, color='gray')
    ax.text(460, 315, 'ESRGAN 4x enhanced', ha='center', fontsize=8, color='#dc2626')

plt.suptitle('TerraWatch — Damage Assessment Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nResults saved to {BASE_DIR}/results_comparison.png')

In [ ]:
# ── EXPORT FOR SERVING ────────────────────────────────────────────────────────
# Save best model weights for FastAPI backend

best_weights = MODELS_DIR / 'enhanced_yolov8n' / 'weights' / 'best.pt'
export_path  = BASE_DIR / 'damage_model_best.pt'

if best_weights.exists():
    shutil.copy(best_weights, export_path)
    print(f'Model exported: {export_path}')
    print(f'Size: {export_path.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('Training still running — best.pt not yet available')

# Save metadata for the API
metadata = {
    'model': 'yolov8n-damage',
    'classes': ['no-damage', 'minor-damage', 'major-damage', 'destroyed'],
    'input_size': 640,
    'esrgan_scale': 4,

    # --- Detection metrics ---
    'baseline_map50':   round(float(baseline_map50), 4),
    'enhanced_map50':   round(float(enhanced_map50), 4),
    'improvement_pct':  round(float(improvement_map50), 2),

    # ⭐ ADD THESE (F1 + PR metrics)
    'baseline_precision': round(float(baseline_precision), 4),
    'baseline_recall':    round(float(baseline_recall), 4),
    'baseline_f1':        round(float(baseline_f1), 4),

    'enhanced_precision': round(float(enhanced_precision), 4),
    'enhanced_recall':    round(float(enhanced_recall), 4),
    'enhanced_f1':        round(float(enhanced_f1), 4),
    'f1_improvement_pct': round(float(improvement_f1), 2),

    'trained_on': 'xBD building damage dataset' if USE_REAL_DATA else 'synthetic data',
}
with open(BASE_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nMetadata:')
print(json.dumps(metadata, indent=2))

In [ ]:
# ── INFERENCE DEMO ────────────────────────────────────────────────────────────
# Show what the full pipeline produces on unseen test images

COLORS_VIZ = {
    0: (  0, 200,  80),  # no-damage: green
    1: (255, 200,   0),  # minor:     yellow
    2: (255, 100,   0),  # major:     orange
    3: (220,  30,  30),  # destroyed: red
}

def run_full_pipeline(img_path, model, upsampler, conf=0.35):
    """Run ESRGAN → YOLOv8n → annotated output."""
    img_bgr = cv2.imread(str(img_path))

    # Enhance
    try:
        enhanced, _ = upsampler.enhance(img_bgr, outscale=4)
        enhanced = cv2.resize(enhanced, (640, 640))
    except Exception:
        enhanced = cv2.resize(img_bgr, (640, 640))

    # Detect
    results = model(enhanced, conf=conf, verbose=False)[0]

    # Draw boxes
    viz = enhanced.copy()
    class_names = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
    damage_counts = {c: 0 for c in class_names}

    for box in results.boxes:
        cls  = int(box.cls[0])
        score_conf = float(box.conf[0])   # ← FIX: avoid overwriting function arg
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        color = COLORS_VIZ[cls]
        damage_counts[class_names[cls]] += 1

        cv2.rectangle(viz, (x1, y1), (x2, y2), color, 2)
        label = f'{class_names[cls]} {score_conf:.2f}'
        cv2.putText(viz, label, (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    # Compute damage score
    total = sum(damage_counts.values())
    weights = {
        'no-damage': 0,
        'minor-damage': 0.33,
        'major-damage': 0.67,
        'destroyed': 1.0
    }
    damage_score = sum(weights[k] * v for k, v in damage_counts.items()) / max(total, 1)

    return cv2.cvtColor(viz, cv2.COLOR_BGR2RGB), damage_counts, damage_score


# Demo on test images
test_images = list((dataset_dir / 'test' / 'images').glob('*.jpg'))[:3] + \
              list((dataset_dir / 'test' / 'images').glob('*.png'))[:3]

best_model = YOLO(str(export_path) if export_path.exists() else 'yolov8n.pt')

# ---- Display evaluated model quality (computed earlier) ----
metrics_badge = f"mAP50={enhanced_map50:.3f} | F1={enhanced_f1:.3f}"

fig, axes = plt.subplots(1, min(3, len(test_images)), figsize=(18, 6))
if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images[:3]):
    viz_img, counts, score = run_full_pipeline(img_path, best_model, upsampler)
    ax.imshow(viz_img)

    risk = 'LOW' if score < 0.3 else 'MODERATE' if score < 0.6 else 'HIGH'
    color = '#16a34a' if risk == 'LOW' else '#d97706' if risk == 'MODERATE' else '#dc2626'

    ax.set_title(f'Damage Score: {score:.2f} — {risk} RISK',
                 fontweight='bold', color=color)

    ax.set_xticks([]); ax.set_yticks([])

    summary = ' | '.join([f'{k[:3]}:{v}' for k, v in counts.items() if v > 0])
    ax.set_xlabel(summary, fontsize=9, color='gray')

plt.suptitle(
    f'TerraWatch — Full Pipeline Inference Demo\n({metrics_badge})',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig(BASE_DIR / 'inference_demo.png', dpi=150, bbox_inches='tight')
plt.show()